# logit-thermo A2-5：データ集め（Pythia の大きさと学びの軸、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 3.5〜4 時間（寝ているあいだに終わって、自動で切れます）。
- ゲート：`docs/A2_5_GATE.md` v0.2。Pythia の 70M・160M・410M・1B・1.4B・2.8B と、1B の学習途中（step 8000・24000・64000）の 9 回。いちばん最後が 2.8B（メモリのため 20 本ずつ 2 回に分けて生成）。
- 記事は A2-2 と同じ 40 本（A2-2 の種で選ぶ）。生成の種だけ新しい（20261102）。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_5_collect.py`（`19b523654033b24c…`）は埋め込み。**logit-thermo フォルダの前のノートブックは動かさないでください。**
- 保存先 `A2_5/`。途中で切れても、もう一度「すべて実行」すれば終わった回は飛ばします。
- **通知**：🚀・回ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-5 データ集め（Pythia の大きさと学びの軸、9 回、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-5 データ集め（Pythia の大きさと学びの軸、9 回、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-5 collect', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'})
SRC22 = _s['SRC22']
SRC5C = '"""A2-5（ゲート docs/A2_5_GATE.md v0.2）のデータ集め。GPU（L4）のノートブックで、scripts/a2_2_collect.py を実行したのと\n同じ名前空間で実行する。A2-2 の run_model と generate は変えずに使い、名前で呼ばれる部品だけを差し替える：\n- load_model を、revision（学習途中のチェックポイント）を受け取る版に差し替える。\n- 生成のあいだだけ SEED を A5_SEED にする（記事は先に A2-2 の種で選んでおく。load_docs も SEED を使うため）。\n- 2.8B だけ、generate を「20 本ずつ 2 回に分けて呼び、つなぐ」版に差し替える（メモリのため）。\n"""\nimport hashlib, json, time\nimport numpy as np\nimport torch\n\nA5_SEED = 20261102\nA5_RUNS = [  # (名前, Hugging Face の ID, revision, 生成を分ける本数（None なら分けない）)\n    (\'pythia-70m\', \'EleutherAI/pythia-70m\', \'main\', None),\n    (\'pythia-160m\', \'EleutherAI/pythia-160m\', \'main\', None),\n    (\'pythia-410m\', \'EleutherAI/pythia-410m\', \'main\', None),\n    (\'pythia-1b\', \'EleutherAI/pythia-1b\', \'main\', None),\n    (\'pythia-1.4b\', \'EleutherAI/pythia-1.4b\', \'main\', None),\n    (\'pythia-1b@step8000\', \'EleutherAI/pythia-1b\', \'step8000\', None),\n    (\'pythia-1b@step24000\', \'EleutherAI/pythia-1b\', \'step24000\', None),\n    (\'pythia-1b@step64000\', \'EleutherAI/pythia-1b\', \'step64000\', None),\n    (\'pythia-2.8b\', \'EleutherAI/pythia-2.8b\', \'main\', 20),\n]\n_A22_SEED = None\n_ORIG_GENERATE = None\n\n\ndef split_spec(spec):\n    return tuple(spec.split(\'@\', 1)) if \'@\' in spec else (spec, None)\n\n\ndef load_model_rev(spec, device):\n    """spec：\'Hugging Face の ID\' または \'ID@revision\'。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id, rev = split_spec(spec)\n    tok = AutoTokenizer.from_pretrained(hf_id, revision=rev)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, revision=rev, torch_dtype=torch.float32).to(device).eval()\n    return tok, model\n\n\ndef make_chunked(orig, size):\n    def generate_chunked(model, prompts, T, valid, gen, H2):\n        outs = [orig(model, prompts[a:a + size], T, valid, gen, H2) for a in range(0, len(prompts), size)]\n        return tuple(np.concatenate([o[q] for o in outs], 0) for q in range(len(outs[0])))\n    return generate_chunked\n\n\ndef pairs_hash(tok, docs):\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    h = hashlib.sha256()\n    for p in pairs:\n        h.update(json.dumps(None if p is None else [list(map(int, p[0])), list(map(int, p[1]))]).encode())\n    return h.hexdigest()\n\n\ndef run_a25(docs, outdir, device, notify=print):\n    """docs は先に load_docs()（A2-2 の種）で読んでおくこと。"""\n    global SEED, load_model, generate, _A22_SEED, _ORIG_GENERATE\n    from transformers import AutoTokenizer\n    _A22_SEED, _ORIG_GENERATE = SEED, generate\n    # 確かめ 4：トークナイザーの区切り方が全部の回で同じ\n    hs = {name: pairs_hash(AutoTokenizer.from_pretrained(hf_id, revision=rev), docs) for name, hf_id, rev, _ in A5_RUNS}\n    assert len(set(hs.values())) == 1, (\'トークナイザーの区切り方が違う回があります\', hs)\n    meta = {\'gen_seed\': A5_SEED, \'doc_seed\': _A22_SEED, \'pairs_sha256\': list(hs.values())[0], \'runs\': {}}\n    load_model = load_model_rev\n    try:\n        for name, hf_id, rev, chunk in A5_RUNS:\n            t0 = time.time()\n            SEED = A5_SEED\n            generate = make_chunked(_ORIG_GENERATE, chunk) if chunk else _ORIG_GENERATE\n            assert run_model.__globals__[\'SEED\'] == A5_SEED and run_model.__globals__[\'load_model\'] is load_model_rev\n            try:\n                run_model(name, f\'{hf_id}@{rev}\', docs, outdir, device, notify=notify)\n            finally:\n                SEED, generate = _A22_SEED, _ORIG_GENERATE\n            Z = np.load(f\'{outdir}/{name}.npz\')\n            meta[\'runs\'][name] = {\'hf_id\': hf_id, \'revision_requested\': rev, \'commit\': str(Z[\'revision\']), \'chunk\': chunk,\n                                  \'use\': [int(u) for u in Z[\'use\']], \'seconds\': round(time.time() - t0, 1)}\n            json.dump(meta, open(f\'{outdir}/a2_5_runs.json\', \'w\'), ensure_ascii=False, indent=1)\n    finally:\n        SEED, generate = _A22_SEED, _ORIG_GENERATE\n    uses = {tuple(r[\'use\']) for r in meta[\'runs\'].values()}\n    assert len(uses) == 1, \'使った記事が回によって違います\'\n    return meta\n\n\ndef self_test_a25(device):\n    assert split_spec(\'EleutherAI/pythia-1b@step8000\') == (\'EleutherAI/pythia-1b\', \'step8000\')\n    assert split_spec(\'EleutherAI/pythia-1b\') == (\'EleutherAI/pythia-1b\', None)\n    tok, model = load_model_rev(\'hf-internal-testing/tiny-random-GPT2LMHeadModel@main\', device)\n    valid, _ = valid_mask(tok, model, device)\n    # 分けて生成しても、形と値の範囲が同じ（同じ生成器を順に使う）\n    prompts = [list(range(5 + i, 5 + i + 8)) for i in range(5)]\n    global CONT\n    keep = CONT; CONT = 6\n    try:\n        g1 = torch.Generator(device=device).manual_seed(0)\n        full = generate(model, prompts, 1.0, valid, g1, 33)\n        g2 = torch.Generator(device=device).manual_seed(0)\n        part = make_chunked(generate, 2)(model, prompts, 1.0, valid, g2, 33)\n    finally:\n        CONT = keep\n    assert all(a.shape == b.shape for a, b in zip(full, part)), [(a.shape, b.shape) for a, b in zip(full, part)]\n    assert np.allclose(full[0][:2, 0], part[0][:2, 0], atol=1e-4)      # 1 歩目は乱数に関係なく同じ\n    assert len({n for n, _, _, _ in A5_RUNS}) == len(A5_RUNS)\n    assert \'SEED\' in run_model.__globals__ and \'generate\' in run_model.__globals__\n    return \'OK\'\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRC5C.encode()).hexdigest() == '19b523654033b24ce50ede75922d4a5ceeff0dbfa59b8a197d4920e1b19db8c8'
exec(SRC22)
exec(SRC5C)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 の種で選び、A2-2 と同じか確かめる）→ 9 回 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test('cuda'), self_test_a25('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_5', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    m = run_a25(docs, '/content/drive/MyDrive/logit-thermo/A2_5', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    assert SEED == 20261027, '種が戻っていません'
    m.update(gate='docs/A2_5_GATE.md v0.2', a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', a2_5_collect_sha256='19b523654033b24ce50ede75922d4a5ceeff0dbfa59b8a197d4920e1b19db8c8',
             torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
             gpu=torch.cuda.get_device_name(0), float32_matmul_precision=torch.get_float32_matmul_precision())
    json.dump(m, open('/content/drive/MyDrive/logit-thermo/A2_5' + '/run_meta.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-5 のデータ集めで止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-5 の 9 回を A2_5 に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
